# Modèle de référence en Machine Learning classique, séparation par sujets exclus (Held-Out-Subject)
### Tous les sujets regroupés, évalués sur des sujets que le modèle n'a jamais vus à l'entraînement

Ce notebook reprend le même modèle et les mêmes comparaisons que la version réalisée sur le sujet 1 seul (modèle fusionné EEG+fNIRS+SOA vs EEG seul vs fNIRS seul, importance des variables en début vs fin d'expérience), mais cette fois entraîné sur `level2_features_all_subjects.csv` et évalué avec une séparation **par sujets exclus** (`GroupShuffleSplit` / `GroupKFold`, regroupés par `subject_id`) plutôt qu'une séparation aléatoire des lignes — de sorte qu'aucun sujet n'apparaisse à la fois dans l'ensemble d'entraînement et de test. C'est la version rigoureuse (honnête) de ce test.

Le résultat obtenu avec séparation par sujets exclus et l'ancien résultat de type intra-sujet sont présentés côte à côte, pour montrer concrètement à quel point le chiffre obtenu sur un seul sujet surestimait la capacité de généralisation du modèle.

In [1]:
import os
DATA_DIR = os.environ.get('EEG_FNIRS_DATA_DIR', 'data')  # folder holding level2_features_all_subjects.csv
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv(os.path.join(DATA_DIR, "level2_features_all_subjects.csv"))
df = df.dropna(subset=['theta', 'alpha', 'beta', 'alpha_beta_ratio', 'spectral_entropy',
                        'frn', 'mean_hbo', 'mean_hbr', 'soa', 'hazard', 'rt1'])
print(f"Loaded {len(df)} trials from {df['subject_id'].nunique()} subjects "
      f"(after dropping rows with missing features)")
df.groupby('subject_id').size().describe()

Loaded 48397 trials from 48 subjects (after dropping rows with missing features)


count      48.000000
mean     1008.270833
std        22.677892
min       910.000000
25%      1010.000000
50%      1016.000000
75%      1020.000000
max      1020.000000
dtype: float64

## 1. Séparation par sujets exclus (le test rigoureux)

`GroupShuffleSplit`, en utilisant `subject_id` comme groupe, garantit que tous les essais d'un même sujet se retrouvent entièrement dans l'ensemble d'entraînement OU entièrement dans l'ensemble de test, jamais dans les deux à la fois.

In [2]:
EEG_FEATS = ['theta', 'alpha', 'beta', 'alpha_beta_ratio', 'spectral_entropy', 'frn']
NIR_FEATS = ['mean_hbo', 'mean_hbr']
SOA_FEAT = ['soa']
TARGET = 'rt1'  # real, measured reaction time -- not a constructed proxy
GROUP = 'subject_id'

fused_feats = EEG_FEATS + NIR_FEATS + SOA_FEAT
X = df[fused_feats].values
y = df[TARGET].values
groups = df[GROUP].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

train_subjects = sorted(df.iloc[train_idx][GROUP].unique())
test_subjects = sorted(df.iloc[test_idx][GROUP].unique())
print(f"Train subjects ({len(train_subjects)}): {train_subjects}")
print(f"Test subjects  ({len(test_subjects)}): {test_subjects}")
assert set(train_subjects).isdisjoint(test_subjects), "Subject leaked across train/test!"

rf_fused = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42)
rf_fused.fit(X[train_idx], y[train_idx])

train_r2 = r2_score(y[train_idx], rf_fused.predict(X[train_idx]))
test_r2_heldout = r2_score(y[test_idx], rf_fused.predict(X[test_idx]))
test_mae_heldout = mean_absolute_error(y[test_idx], rf_fused.predict(X[test_idx]))

print(f"\nHeld-out-subject: train R2={train_r2:.3f}  test R2={test_r2_heldout:.3f}  "
      f"test MAE={test_mae_heldout:.3f}")

Train subjects (38): [np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(7), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(41), np.int64(42), np.int64(44), np.int64(45), np.int64(47), np.int64(48), np.int64(49), np.int64(50)]
Test subjects  (10): [np.int64(6), np.int64(15), np.int64(22), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(40), np.int64(43), np.int64(46)]

Held-out-subject: train R2=0.100  test R2=-0.001  test MAE=1.118


## 2. R² par validation croisée avec sujets exclus (plus stable qu'une seule séparation)

Avec un nombre limité de sujets, le fait qu'un sujet donné se retrouve ou non dans l'ensemble de test peut faire beaucoup varier le résultat. `GroupKFold` répète plusieurs fois la séparation par sujets exclus et fait la moyenne des résultats obtenus.

In [3]:
n_subjects = df[GROUP].nunique()
n_splits = min(5, n_subjects)  # can't have more folds than subjects

gkf = GroupKFold(n_splits=n_splits)
cv_r2_scores = []
for tr_idx, te_idx in gkf.split(X, y, groups):
    m = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42)
    m.fit(X[tr_idx], y[tr_idx])
    cv_r2_scores.append(r2_score(y[te_idx], m.predict(X[te_idx])))

cv_r2_scores = np.array(cv_r2_scores)
print(f"{n_splits}-fold GroupKFold (held-out subjects each fold):")
print(f"  R2 per fold: {np.round(cv_r2_scores, 3)}")
print(f"  Mean R2: {cv_r2_scores.mean():.3f}  +/- {cv_r2_scores.std():.3f}")

5-fold GroupKFold (held-out subjects each fold):
  R2 per fold: [-4.0419e+02 -2.0459e+02 -6.5706e+02 -1.0000e-03 -3.0376e+02]
  Mean R2: -313.920  +/- 217.483


## 3. Comparaison : séparation par sujets exclus vs séparation naïve de type intra-sujet

L'ancien notebook (limité au sujet 1) utilisait une séparation aléatoire des lignes, ce qui permettait aux essais d'un même sujet de se retrouver à la fois dans l'entraînement et dans le test. En reproduisant ici cette même séparation naïve, mais sur l'ensemble des sujets regroupés, on voit directement à quel point elle surestimait la capacité de généralisation du modèle.

In [4]:
Xtr_naive, Xte_naive, ytr_naive, yte_naive = train_test_split(X, y, test_size=0.2, random_state=42)
rf_naive = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42)
rf_naive.fit(Xtr_naive, ytr_naive)
test_r2_naive = r2_score(yte_naive, rf_naive.predict(Xte_naive))

print(f"Naive random row split (subjects can leak across train/test): test R2 = {test_r2_naive:.3f}")
print(f"Held-out-subject split (single split):                        test R2 = {test_r2_heldout:.3f}")
print(f"Held-out-subject split (5-fold mean):                         test R2 = {cv_r2_scores.mean():.3f}")
print(f"\nGap (naive - honest): {test_r2_naive - cv_r2_scores.mean():+.3f} "
      f"-- positive means the naive split was overstating performance")

Naive random row split (subjects can leak across train/test): test R2 = -0.043
Held-out-subject split (single split):                        test R2 = -0.001
Held-out-subject split (5-fold mean):                         test R2 = -313.920

Gap (naive - honest): +313.877 -- positive means the naive split was overstating performance


## 4. EEG seul vs fNIRS seul vs fusion — version avec sujets exclus

In [5]:
def fit_eval_grouped(feat_cols, label):
    Xf = df[feat_cols].values
    scores = []
    for tr_idx, te_idx in gkf.split(Xf, y, groups):
        m = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42)
        m.fit(Xf[tr_idx], y[tr_idx])
        scores.append(r2_score(y[te_idx], m.predict(Xf[te_idx])))
    scores = np.array(scores)
    print(f"{label:22s} mean R2={scores.mean():.3f}  +/- {scores.std():.3f}  (per fold: {np.round(scores, 3)})")
    return scores.mean()

r2_eeg = fit_eval_grouped(EEG_FEATS + SOA_FEAT, "EEG + SOA only")
r2_nir = fit_eval_grouped(NIR_FEATS + SOA_FEAT, "fNIRS + SOA only")
r2_fused = fit_eval_grouped(fused_feats, "Fused (EEG+fNIRS+SOA)")

print(f"\nFusion gain over best single modality (held-out-subject): "
      f"{r2_fused - max(r2_eeg, r2_nir):+.3f} R2")

EEG + SOA only         mean R2=-313.433  +/- 215.947  (per fold: [-4.04009e+02 -2.06227e+02 -6.52760e+02 -1.00000e-03 -3.04166e+02])
fNIRS + SOA only       mean R2=-479.642  +/- 328.407  (per fold: [-2.49169e+02 -4.99658e+02 -9.18406e+02 -1.00000e-03 -7.30976e+02])
Fused (EEG+fNIRS+SOA)  mean R2=-313.920  +/- 217.483  (per fold: [-4.0419e+02 -2.0459e+02 -6.5706e+02 -1.0000e-03 -3.0376e+02])

Fusion gain over best single modality (held-out-subject): -0.488 R2


## 5. La véritable comparaison de validation — les données cérébrales apportent-elles quelque chose de plus que la théorie seule ?

`hazard` est une fonction déterministe de `soa` (calculée une seule fois au Niveau 1+2, à partir de la distribution regroupée des SOA). Un modèle qui prédit le RT uniquement à partir de `hazard` n'utilise **aucune donnée EEG/fNIRS** — il représente « dans quelle mesure peut-on prédire le RT en connaissant seulement le design expérimental », c'est-à-dire le plancher que tout modèle réellement basé sur les données cérébrales doit dépasser.

C'est la version rigoureuse de la demande initiale (tâche 4) de valider que les grandeurs estimées se comportent conformément à la théorie : il ne s'agit pas seulement de vérifier qu'une corrélation existe, mais de vérifier que le modèle fusionné EEG+fNIRS+SOA explique **davantage** de variance que le design expérimental seul, en utilisant la même méthode d'évaluation par sujets exclus que partout ailleurs dans ce notebook.

In [6]:
def fit_eval_grouped_generic(feat_cols, label):
    Xf = df[feat_cols].values
    scores = []
    for tr_idx, te_idx in gkf.split(Xf, y, groups):
        m = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42)
        m.fit(Xf[tr_idx], y[tr_idx])
        scores.append(r2_score(y[te_idx], m.predict(Xf[te_idx])))
    scores = np.array(scores)
    print(f"{label:30s} mean R2={scores.mean():.3f}  +/- {scores.std():.3f}  (per fold: {np.round(scores, 3)})")
    return scores.mean(), scores

r2_hazard_only, _ = fit_eval_grouped_generic(['hazard'], "Hazard(SOA) only -- theory floor")
r2_soa_only, _ = fit_eval_grouped_generic(SOA_FEAT, "Raw SOA only")

print(f"\nFused model (EEG+fNIRS+SOA, from section 4):        R2 = {r2_fused:.3f}")
print(f"Hazard-only theoretical floor:                        R2 = {r2_hazard_only:.3f}")
print(f"Gain of fused model over theory-alone floor:          {r2_fused - r2_hazard_only:+.3f}")
if r2_fused > r2_hazard_only:
    print("\\nPASS -- brain data explains more than the design alone")
else:
    print("\\nBrain data does NOT currently beat the design-alone floor -- fused model is not adding real information yet")


Hazard(SOA) only -- theory floor mean R2=-5.838  +/- 2.998  (per fold: [-6.647e+00 -7.251e+00 -8.559e+00 -1.000e-03 -6.732e+00])
Raw SOA only                   mean R2=-511.878  +/- 322.608  (per fold: [-3.47202e+02 -5.58659e+02 -9.50756e+02 -1.00000e-03 -7.02770e+02])
\nFused model (EEG+fNIRS+SOA, from section 4):        R2 = -313.920
Hazard-only theoretical floor:                        R2 = -5.838
Gain of fused model over theory-alone floor:          -308.082
\nBrain data does NOT currently beat the design-alone floor -- fused model is not adding real information yet


## 6. Early-vs-late trials feature importance (pooled across subjects)

In [7]:
median_row = df.groupby(['subject_id', 'block'])['trial_row'].transform('median')
df['phase'] = np.where(df['trial_row'] <= median_row, 'early', 'late')

importances = {}
for phase in ['early', 'late']:
    sub = df[df['phase'] == phase]
    Xp = sub[fused_feats].values
    yp = sub[TARGET].values
    m = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42)
    m.fit(Xp, yp)
    importances[phase] = dict(zip(fused_feats, m.feature_importances_))

imp_df = pd.DataFrame(importances)
imp_df['delta_late_minus_early'] = imp_df['late'] - imp_df['early']
imp_df.sort_values('delta_late_minus_early', ascending=False)

,early,late,delta_late_minus_early
spectral_entropy,0.040010,0.345269,0.305259
beta,0.068790,0.179752,0.110962
theta,0.134671,0.232021,0.097351
frn,0.044856,0.098736,0.053880
alpha_beta_ratio,0.032663,0.009678,-0.022986
soa,0.173039,0.093799,-0.079240
alpha,0.118546,0.034071,-0.084474
mean_hbr,0.179064,0.003350,-0.175715
mean_hbo,0.208361,0.003324,-0.205037


## 6. Résumé

In [8]:
print("="*60)
print("BATCH LEVEL 3 SUMMARY - All Subjects, Held-Out-Subject Test")
print("="*60)
print(f"Subjects: {n_subjects}  |  Total trials: {len(df)}")
print(f"\nFused model:")
print(f"  Naive row-split test R2:            {test_r2_naive:.3f}")
print(f"  Held-out-subject test R2 (1 split):  {test_r2_heldout:.3f}")
print(f"  Held-out-subject test R2 ({n_splits}-fold mean): {cv_r2_scores.mean():.3f} +/- {cv_r2_scores.std():.3f}")
print(f"\nEEG-only R2={r2_eeg:.3f}  fNIRS-only R2={r2_nir:.3f}  Fused R2={r2_fused:.3f}")
pass_metrics = cv_r2_scores.mean() > 0
print(f"\nPasses target metric (held-out-subject R2 > 0): {pass_metrics}")
print("\nFeature importance shift (early vs late trials, pooled):")
print(imp_df.sort_values('delta_late_minus_early', ascending=False).to_string())

BATCH LEVEL 3 SUMMARY - All Subjects, Held-Out-Subject Test
Subjects: 48  |  Total trials: 48397

Fused model:
  Naive row-split test R2:            -0.043
  Held-out-subject test R2 (1 split):  -0.001
  Held-out-subject test R2 (5-fold mean): -313.920 +/- 217.483

EEG-only R2=-313.433  fNIRS-only R2=-479.642  Fused R2=-313.920

Passes target metric (held-out-subject R2 > 0): False

Feature importance shift (early vs late trials, pooled):
                     early      late  delta_late_minus_early
spectral_entropy  0.040010  0.345269                0.305259
beta              0.068790  0.179752                0.110962
theta             0.134671  0.232021                0.097351
frn               0.044856  0.098736                0.053880
alpha_beta_ratio  0.032663  0.009678               -0.022986
soa               0.173039  0.093799               -0.079240
alpha             0.118546  0.034071               -0.084474
mean_hbr          0.179064  0.003350               -0.175715
mean_hbo